# 03 - ETL gobernado, cuarentena y carga atomica

## Objetivo y entorno

Demostrar un ETL local reproducible que preserva la fuente, registra hallazgos, bloquea lotes no aptos, carga con integridad y evita duplicados. Se usan Python, pandas y `sqlite3`, incluido en la biblioteca estandar. SQLite es una base embebida: no hay servidor ni plataforma externa. Cada ejecucion utiliza una base en memoria y comienza desde cero; la persistencia de produccion requiere archivo o servidor, controles de acceso, respaldo y retencion.

## Paso 1 - Conocer el flujo y sus limites

`ejecutar_etl` normaliza y evalua antes de publicar. Un lote recibe identificador, huella de fuente, version del contrato, estado y conteos. Los hallazgos conservan `fila_fuente`, regla y accion. El dataframe `rechazadas` conserva filas en cuarentena durante la ejecucion, pero no se persisten sus contenidos completos en SQLite: solo sus hallazgos. Una cuarentena durable debe implementarse con proteccion, retencion y permisos antes de produccion.

La tabla de ventas tiene PK por id, FK al maestro y restricciones numericas y de moneda. `PRAGMA foreign_keys=ON` activa las FK de SQLite. Se carga dentro de una transaccion y se reconcilian los importes por moneda: sumar ARS y USD en un unico numero no tiene significado sin conversion aprobada.

El lote defectuoso de 12 filas tiene 8 rechazadas; supera la puerta del 20%. Aunque 4 filas sean aptas, el lote completo queda BLOQUEADO y no se publica ninguna venta. Los hallazgos y el estado si se registran.

Luego se prepara un **subconjunto de prueba apto** con las 4 filas sin fallos de cuarentena. No es una reparacion del lote original: las 8 filas restantes siguen pendientes de investigacion. Publicar solamente ese subconjunto en un proceso real requeriria alcance y decision autorizados. Esta distincion evita simular calidad eliminando registros sin evidencia.

### Organizacion del codigo

La siguiente celda muestra toda la implementacion especifica de esta practica: puerta de publicacion, huella, creacion de SQLite y funcion `ejecutar_etl`. El DDL, las transacciones, la resolucion de conflictos, la reconciliacion y el registro del lote pueden leerse aqui.

El ETL llama a `demo.normalizar` y `demo.evaluar`, que pertenecen al nucleo usado por los cinco notebooks. El resto de la carga es local. La etiqueta `ERROR_TECNICO` es una simplificacion de esta implementacion didactica; no se ha cambiado la politica ni el comportamiento al trasladar las funciones.

In [ ]:
import hashlib
import sqlite3
from uuid import uuid4

import pandas as pd


def puerta_publicacion(total, rechazadas, max_rechazo=0.20):
    if not 0 <= max_rechazo <= 1 or total < 0 or not 0 <= rechazadas <= total:
        raise ValueError('Parametros invalidos para la puerta de publicacion')
    return total > 0 and rechazadas < total and rechazadas / total <= max_rechazo


def huella(datos):
    contenido = datos.to_json(orient='split', date_format='iso', index=False)
    return hashlib.sha256(contenido.encode('utf-8')).hexdigest()


def nueva_conexion():
    conexion = sqlite3.connect(':memory:')
    conexion.execute('PRAGMA foreign_keys = ON')
    conexion.executescript('''
        CREATE TABLE productos (id_producto INTEGER PRIMARY KEY, producto TEXT NOT NULL);
        CREATE TABLE ventas (
            id_venta INTEGER PRIMARY KEY,
            id_producto INTEGER NOT NULL REFERENCES productos(id_producto),
            fecha_venta TEXT NOT NULL,
            monto_centavos INTEGER NOT NULL CHECK (monto_centavos > 0),
            moneda TEXT NOT NULL CHECK (moneda IN ('ARS', 'USD', 'EUR')),
            fila_fuente TEXT NOT NULL,
            lote_inicial TEXT NOT NULL
        );
        CREATE TABLE lotes (
            lote TEXT PRIMARY KEY, huella TEXT NOT NULL, contrato TEXT NOT NULL,
            estado TEXT NOT NULL, total INTEGER NOT NULL, aceptadas INTEGER NOT NULL,
            rechazadas INTEGER NOT NULL, nuevas INTEGER NOT NULL, existentes INTEGER NOT NULL
        );
        CREATE TABLE hallazgos (
            lote TEXT NOT NULL REFERENCES lotes(lote), fila_fuente TEXT NOT NULL,
            regla TEXT NOT NULL, accion TEXT NOT NULL
        );
    ''')
    return conexion


def ejecutar_etl(datos_crudos, productos, referencia, conexion):
    datos = demo.normalizar(datos_crudos)
    aceptadas, rechazadas, metricas, detalle = demo.evaluar(datos, productos, referencia)
    lote = uuid4().hex
    estado = 'PUBLICADO' if puerta_publicacion(len(datos), len(rechazadas)) else 'BLOQUEADO'
    nuevas, existentes = 0, 0
    try:
        with conexion:
            if estado == 'PUBLICADO':
                for producto in productos.itertuples(index=False):
                    existente = conexion.execute('SELECT producto FROM productos WHERE id_producto=?', (int(producto.id_producto),)).fetchone()
                    if existente is None:
                        conexion.execute('INSERT INTO productos VALUES (?, ?)', (int(producto.id_producto), producto.producto))
                    elif existente[0] != producto.producto:
                        raise ValueError('Cambio de maestro: requiere politica SCD aprobada')
                for venta in aceptadas.itertuples(index=False):
                    valores = (int(venta.id_producto), venta.fecha_venta.date().isoformat(), int(venta.monto_centavos), str(venta.moneda))
                    existente = conexion.execute('SELECT id_producto, fecha_venta, monto_centavos, moneda FROM ventas WHERE id_venta=?', (int(venta.id_venta),)).fetchone()
                    if existente is None:
                        conexion.execute('INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?, ?)', (int(venta.id_venta), *valores, venta.fila_fuente, lote))
                        nuevas += 1
                    elif existente == valores:
                        existentes += 1
                    else:
                        raise ValueError('Conflicto de contenido para una venta ya publicada')
                claves = [int(identificador) for identificador in aceptadas['id_venta']]
                marcadores = ','.join('?' for _ in claves)
                destino = pd.read_sql_query(f'SELECT id_venta, moneda, monto_centavos FROM ventas WHERE id_venta IN ({marcadores})', conexion, params=claves)
                assert len(destino) == len(aceptadas)
                esperado = {str(moneda): int(total) for moneda, total in aceptadas.groupby('moneda')['monto_centavos'].sum().items()}
                obtenido = {str(moneda): int(total) for moneda, total in destino.groupby('moneda')['monto_centavos'].sum().items()}
                assert esperado == obtenido
            conexion.execute('INSERT INTO lotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', (lote, huella(datos_crudos), demo.VERSION_CONTRATO, estado, len(datos), len(aceptadas), len(rechazadas), nuevas, existentes))
            conexion.executemany('INSERT INTO hallazgos VALUES (?, ?, ?, ?)', [(lote, fila.fila_fuente, fila.regla, fila.accion) for fila in detalle.itertuples(index=False)])
    except Exception:
        with conexion:
            conexion.execute('INSERT INTO lotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)', (lote, huella(datos_crudos), demo.VERSION_CONTRATO, 'ERROR_TECNICO', len(datos), len(aceptadas), len(rechazadas), 0, 0))
        raise
    return {'lote': lote, 'estado': estado, 'nuevas': nuevas, 'existentes': existentes, 'aceptadas': aceptadas, 'rechazadas': rechazadas, 'metricas': metricas, 'detalle': detalle}

In [ ]:
import sys
from pathlib import Path

import pandas as pd

candidatos = [Path.cwd(), Path.cwd() / 'notebook' / 'unidad_III']
carpeta = next((ruta for ruta in candidatos if (ruta / 'gobierno_demo.py').exists()), None)
if carpeta is None:
    raise RuntimeError('Ejecutar desde la raiz del repositorio o notebook/unidad_III')
sys.path.insert(0, str(carpeta.resolve()))
import gobierno_demo as demo

crudos, productos, referencia = demo.crear_datos()
conexion = nueva_conexion()
assert conexion.execute('PRAGMA foreign_keys').fetchone()[0] == 1
bloqueado = ejecutar_etl(crudos, productos, referencia, conexion)
assert bloqueado['estado'] == 'BLOQUEADO'
assert conexion.execute('SELECT COUNT(*) FROM ventas').fetchone()[0] == 0
display(bloqueado['detalle'])

ids_aptos = set(bloqueado['aceptadas']['fila_fuente'])
subconjunto = crudos[crudos['fila_fuente'].isin(ids_aptos)].copy()
publicado = ejecutar_etl(subconjunto, productos, referencia, conexion)
assert publicado['estado'] == 'PUBLICADO'
assert publicado['nuevas'] == 4
display(pd.read_sql_query('SELECT * FROM ventas', conexion))
display(pd.read_sql_query('SELECT estado, total, aceptadas, rechazadas, nuevas, existentes FROM lotes', conexion))
display(pd.read_sql_query('SELECT moneda, SUM(monto_centavos) AS total_centavos FROM ventas GROUP BY moneda', conexion))
assert 'email' not in [campo[1] for campo in conexion.execute('PRAGMA table_info(ventas)')]
print('La venta se publica sin email; el lote original permanece bloqueado.')

## Paso 2 - Idempotencia y rollback

Una recarga identica debe registrar otro intento, pero no insertar otra venta ni cambiar su contenido. **Idempotencia del dato no implica eliminar la auditoria de los intentos.** Las 4 filas publicadas se conservan y la nueva corrida informa 4 existentes y 0 nuevas.

Se prepara luego un lote de prueba con dos filas: una venta nueva y una venta ya publicada cuyo importe se cambia intencionalmente. La politica del ejemplo no permite sobrescribir una venta publicada: detecta conflicto y aborta. Como ambas filas se procesan en una transaccion, la venta nueva que se intento insertar antes del conflicto tambien debe desaparecer al hacer rollback.

El ejemplo registra el intento fallido con estado `ERROR_TECNICO`, una etiqueta simplificada de la implementacion local. Este conflicto es realmente un error de contenido; una implementacion productiva debe distinguirlo de conexion, timeout o caida de servicio y decidir reintentos diferentes. No se debe reintentar sin investigar un conflicto como si fuera un problema transitorio.

Se comprobaran el contenido antes/despues, la ausencia de la venta nueva y el registro del intento fallido. La base en memoria se cierra al final.

In [ ]:
antes = pd.read_sql_query('SELECT * FROM ventas ORDER BY id_venta', conexion)
recarga = ejecutar_etl(subconjunto, productos, referencia, conexion)
assert recarga['nuevas'] == 0 and recarga['existentes'] == 4
pd.testing.assert_frame_equal(antes, pd.read_sql_query('SELECT * FROM ventas ORDER BY id_venta', conexion))

nueva = subconjunto.iloc[[0]].copy()
nueva.loc[:, 'id_venta'] = '20'
nueva.loc[:, 'fila_fuente'] = 'prueba-nueva'
conflicto = subconjunto.iloc[[0]].copy()
conflicto.loc[:, 'monto'] = '1501.00'
lote_conflictivo = pd.concat([nueva, conflicto], ignore_index=True)
fallo_detectado = False
try:
    ejecutar_etl(lote_conflictivo, productos, referencia, conexion)
except ValueError as error:
    assert 'Conflicto de contenido' in str(error)
    fallo_detectado = True
    print('Conflicto esperado: transaccion revertida.')
assert fallo_detectado
assert conexion.execute('SELECT COUNT(*) FROM ventas WHERE id_venta=20').fetchone()[0] == 0
pd.testing.assert_frame_equal(antes, pd.read_sql_query('SELECT * FROM ventas ORDER BY id_venta', conexion))
display(pd.read_sql_query('SELECT estado, total, nuevas, existentes FROM lotes', conexion))
assert conexion.execute("SELECT COUNT(*) FROM lotes WHERE estado='ERROR_TECNICO'").fetchone()[0] == 1
conexion.close()
print('Idempotencia y rollback verificados.')

## Resultados esperados y extensiones

- Primera corrida: lote original bloqueado, 0 ventas publicadas y hallazgos registrados.
- Subconjunto apto: 4 ventas insertadas sin email en el destino analitico.
- Recarga: 0 nuevas, 4 existentes y contenido sin cambios.
- Conflicto: rollback de todo el lote, venta 20 ausente y datos anteriores preservados.

### Ejercicios

1. Sustituir `:memory:` por un archivo SQLite propio y probar que la recarga sigue sin duplicar entre sesiones. Definir como se protege y elimina el archivo.
2. Implementar una cuarentena persistente minimizada con lote, fila, reglas, estado de revision y plazo de retencion. No guardar PII en logs generales.
3. Separar incidentes de contenido y errores tecnicos. Proponer cuales permiten reintento automatico y cuales exigen autorizacion.
4. Anadir una reconciliacion por producto y moneda y un control de volumen esperado; demostrar que una extraccion incompleta puede ser valida sintacticamente pero no apta para publicar.

El laboratorio no reemplaza control de concurrencia distribuido, autorizacion institucional ni una politica de correccion de ventas. Una actualizacion autorizada necesita conservar antes/despues y su justificacion.